# 🎯 Auto-Labeling Notebook
**Etiketlenmemiş görüntüleri YOLO modeliyle otomatik etiketle → unilabellm ile gözden geçir**

## Workflow
```
Unlabeled Images  →  Inference  →  YOLO .txt Labels  →  unilabellm Review  →  Export  →  Retrain
```

## Kaggle Inputs
| Slot | Tip | Açıklama |
|------|-----|----------|
| `model` | Model | Kendi eğittiğin `.pt` dosyası (önerilen) |
| `images` | Dataset | Etiketlenecek görüntü klasörü |

In [ ]:
# ── 1. Kurulum ──────────────────────────────────────────────────────────────
!pip install ultralytics --quiet
import ultralytics; ultralytics.checks()

In [ ]:
# ── 2. Konfigürasyon ─────────────────────────────────────────────────────────
from pathlib import Path

# ---------------------------------------------------------------------------
# MODEL SEÇİMİ
# ---------------------------------------------------------------------------
# Seçenek A: Kaggle'a yüklediğin kendi modelini kullan (önerilen)
#   → Kaggle'da sağ panelden Add Input > Models > kendi modelini ekle
# Seçenek B: YOLOWorld-M — zero-shot, text-prompt ile herhangi bir sınıf
#   → MODEL_SOURCE = 'yoloworld'
# Seçenek C: YOLO11m pretrained COCO (80 genel sınıf)
#   → MODEL_SOURCE = 'pretrained'

MODEL_SOURCE = 'custom'   # 'custom' | 'yoloworld' | 'pretrained'

# Seçenek A için: /kaggle/input/<model-slug>/... şeklinde .pt yolu
CUSTOM_MODEL_PATH = ''   # boş bırakırsan otomatik bulur

# Seçenek B için: tespit etmesini istediğin sınıflar (virgülle ayrılmış)
YOLOWORLD_CLASSES = ['tank', 'military vehicle', 'armored vehicle', 'truck', 'person']

# ---------------------------------------------------------------------------
# INFERENCE PARAMETRELERİ
# ---------------------------------------------------------------------------
CONF_THRESHOLD  = 0.30   # düşüt = daha fazla label ama gürültülü; yükselt = temiz ama az
IOU_THRESHOLD   = 0.45   # NMS IoU eşiği
IMG_SIZE        = 640    # inference çözünürlüğü
BATCH_SIZE      = 16     # GPU'ya göre artırabilirsin

# ---------------------------------------------------------------------------
# GİRDİ / ÇIKTI
# ---------------------------------------------------------------------------
# Görüntü dizini — Kaggle input dataset'inden otomatik bulur
IMAGE_INPUT_DIR  = ''    # boş = otomatik
# Çıktı dizini
OUTPUT_DIR       = Path('/kaggle/working/auto_labeled')
# Sadece bu uzantıları işle
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

print('Config OK')

In [ ]:
# ── 3. Model Yükle ───────────────────────────────────────────────────────────
from ultralytics import YOLO

def find_pt_model(base='/kaggle/input'):
    """Kaggle input klasörlerinde ilk .pt dosyasını bul."""
    candidates = sorted(Path(base).rglob('*.pt'))
    if not candidates:
        raise FileNotFoundError(f'Hiç .pt dosyası bulunamadı: {base}')
    print(f'  Bulunan modeller: {[str(c) for c in candidates]}')
    # best.pt veya last.pt öncelikli
    for priority in ['best.pt', 'last.pt']:
        for c in candidates:
            if c.name == priority:
                return c
    return candidates[0]

if MODEL_SOURCE == 'custom':
    model_path = Path(CUSTOM_MODEL_PATH) if CUSTOM_MODEL_PATH else find_pt_model()
    print(f'✅ Custom model yükleniyor: {model_path}')
    model = YOLO(str(model_path))
    CLASS_NAMES = model.names   # {0: 'tank', 1: 'person', ...}

elif MODEL_SOURCE == 'yoloworld':
    print('✅ YOLOWorld-M yükleniyor (zero-shot)...')
    model = YOLO('yolov8m-worldv2.pt')
    model.set_classes(YOLOWORLD_CLASSES)
    CLASS_NAMES = {i: c for i, c in enumerate(YOLOWORLD_CLASSES)}

elif MODEL_SOURCE == 'pretrained':
    print('✅ YOLO11m COCO yükleniyor...')
    model = YOLO('yolo11m.pt')
    CLASS_NAMES = model.names

print(f'\n📦 Sınıflar ({len(CLASS_NAMES)} adet):')
for cid, cname in CLASS_NAMES.items():
    print(f'  [{cid}] {cname}')

In [ ]:
# ── 4. Görüntü Klasörünü Bul ─────────────────────────────────────────────────
import os

def find_image_dir(base='/kaggle/input'):
    """En fazla görüntü içeren alt klasörü bul."""
    best_dir, best_count = Path(base), 0
    for dirpath, dirnames, filenames in os.walk(base):
        imgs = [f for f in filenames if Path(f).suffix.lower() in IMAGE_EXTENSIONS]
        if len(imgs) > best_count:
            best_count = len(imgs)
            best_dir = Path(dirpath)
    return best_dir, best_count

if IMAGE_INPUT_DIR:
    img_dir = Path(IMAGE_INPUT_DIR)
    img_count = sum(1 for f in img_dir.rglob('*') if f.suffix.lower() in IMAGE_EXTENSIONS)
else:
    img_dir, img_count = find_image_dir()

print(f'📂 Görüntü dizini : {img_dir}')
print(f'🖼️  Görüntü sayısı : {img_count:,}')

# Tüm görüntü yollarını listele
all_images = sorted([p for p in img_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTENSIONS])
print(f'\nİlk 5 görüntü:')
for p in all_images[:5]:
    print(f'  {p}')

In [ ]:
# ── 5. Çıktı Klasörü Hazırla ─────────────────────────────────────────────────
import shutil

img_out   = OUTPUT_DIR / 'images'
label_out = OUTPUT_DIR / 'labels'
img_out.mkdir(parents=True, exist_ok=True)
label_out.mkdir(parents=True, exist_ok=True)

print(f'📁 Çıktı: {OUTPUT_DIR}')
print(f'   images/ → görüntü kopyaları')
print(f'   labels/ → YOLO .txt etiket dosyaları')

In [ ]:
# ── 6. Toplu Inference + Label Kaydet ────────────────────────────────────────
import shutil
from collections import defaultdict

stats = {
    'total': len(all_images),
    'labeled': 0,
    'empty': 0,
    'class_counts': defaultdict(int),
    'det_per_image': [],
}

def save_yolo_labels(result, label_path):
    """Ultralytics Result nesnesinden YOLO .txt yaz."""
    lines = []
    if result.boxes is None or len(result.boxes) == 0:
        return 0
    
    orig_h, orig_w = result.orig_shape
    for box in result.boxes:
        cls_id = int(box.cls.item())
        conf   = float(box.conf.item())
        # xyxy → cx cy w h (normalized)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx = ((x1 + x2) / 2) / orig_w
        cy = ((y1 + y2) / 2) / orig_h
        bw = (x2 - x1) / orig_w
        bh = (y2 - y1) / orig_h
        lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        stats['class_counts'][CLASS_NAMES.get(cls_id, f'cls_{cls_id}')] += 1
    
    if lines:
        label_path.write_text('\n'.join(lines))
    return len(lines)


print(f'🚀 {len(all_images):,} görüntü işleniyor... (batch={BATCH_SIZE})')
print(f'   conf={CONF_THRESHOLD}  iou={IOU_THRESHOLD}  imgsz={IMG_SIZE}')
print()

# Batch inference
for i in range(0, len(all_images), BATCH_SIZE):
    batch = all_images[i:i+BATCH_SIZE]
    batch_str = [str(p) for p in batch]
    
    results = model.predict(
        batch_str,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE,
        verbose=False,
        device=0,   # GPU; CPU için device='cpu'
    )
    
    for img_path, result in zip(batch, results):
        # Görüntüyü kopyala
        dest_img = img_out / img_path.name
        shutil.copy2(img_path, dest_img)
        
        # Label kaydet
        label_path = label_out / (img_path.stem + '.txt')
        n_dets = save_yolo_labels(result, label_path)
        stats['det_per_image'].append(n_dets)
        
        if n_dets > 0:
            stats['labeled'] += 1
        else:
            stats['empty'] += 1
    
    # Progress
    done = min(i + BATCH_SIZE, len(all_images))
    pct  = done / len(all_images) * 100
    print(f'  [{done:>5}/{len(all_images)}] {pct:5.1f}%  |  '
          f'labeled: {stats["labeled"]}  empty: {stats["empty"]}')

print('\n✅ Inference tamamlandı!')

In [ ]:
# ── 7. İstatistikler ─────────────────────────────────────────────────────────
import numpy as np

total     = stats['total']
labeled   = stats['labeled']
empty     = stats['empty']
total_det = sum(stats['det_per_image'])
det_arr   = np.array(stats['det_per_image'])

print('=' * 50)
print('📊 AUTO-LABELING SONUÇLARI')
print('=' * 50)
print(f'Toplam görüntü    : {total:,}')
print(f'Etiketlenen       : {labeled:,} ({labeled/total*100:.1f}%)')
print(f'Boş (tespit yok)  : {empty:,} ({empty/total*100:.1f}%)')
print(f'Toplam tespit     : {total_det:,}')
if labeled > 0:
    print(f'Ort. det/görüntü  : {det_arr[det_arr>0].mean():.1f}')
    print(f'Maks. det/görüntü : {det_arr.max()}')
print()
print('─── Sınıf Dağılımı ───')
for cls_name, cnt in sorted(stats['class_counts'].items(), key=lambda x: -x[1]):
    bar = '█' * int(cnt / max(stats['class_counts'].values()) * 30)
    print(f'  {cls_name:<25} {cnt:>6,}  {bar}')
print('=' * 50)

In [ ]:
# ── 8. Örnek Görselleştirme ───────────────────────────────────────────────────
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

PALETTE = [
    '#00e676','#40c4ff','#ff6d00','#ea80fc',
    '#ffd740','#69f0ae','#ff4081','#80d8ff',
    '#b2ff59','#ffd180','#ff9e80','#b388ff',
]

def draw_sample(ax, img_path: Path, label_path: Path):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=7, pad=3)
    ax.axis('off')
    
    if not label_path.exists():
        return
    
    w, h = img.size
    for line in label_path.read_text().strip().splitlines():
        if not line.strip(): continue
        parts = line.split()
        cls_id = int(parts[0])
        cx, cy, bw, bh = map(float, parts[1:5])
        x1 = (cx - bw/2) * w
        y1 = (cy - bh/2) * h
        color = PALETTE[cls_id % len(PALETTE)]
        rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                   linewidth=1.5, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        cls_name = CLASS_NAMES.get(cls_id, f'cls_{cls_id}')
        ax.text(x1, y1-3, cls_name, color=color, fontsize=6,
                bbox=dict(boxstyle='square,pad=0.1', fc='black', alpha=0.7, lw=0))

# Etiketlenmiş görüntülerden rastgele 16 tane seç
labeled_imgs = [p for p in img_out.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS]
labeled_imgs = [p for p in labeled_imgs if (label_out / (p.stem + '.txt')).exists()
                and (label_out / (p.stem + '.txt')).stat().st_size > 0]

sample = random.sample(labeled_imgs, min(16, len(labeled_imgs)))
cols, rows = 4, max(1, (len(sample) + 3) // 4)
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3.5))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, ax in enumerate(axes):
    if i < len(sample):
        draw_sample(ax, sample[i], label_out / (sample[i].stem + '.txt'))
    else:
        ax.axis('off')

plt.suptitle(f'Auto-Label Örnekleri (conf ≥ {CONF_THRESHOLD})', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('preview.png kaydedildi.')

In [ ]:
# ── 9. data.yaml Oluştur ─────────────────────────────────────────────────────
import yaml

# Boş görüntüleri filtrele (opsiyonel — yorum satırını kaldır)
# empty_labels = [f for f in label_out.iterdir() if f.stat().st_size == 0]
# for f in empty_labels: f.unlink(); (img_out / (f.stem + Path(...)...)).unlink()

yaml_content = {
    'path' : str(OUTPUT_DIR),
    'train': 'images',
    'val'  : 'images',   # split yoksa aynı klasör; sonradan bölebilirsin
    'nc'   : len(CLASS_NAMES),
    'names': list(CLASS_NAMES.values()),
}

yaml_path = OUTPUT_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

print('📄 data.yaml:')
print(yaml_path.read_text())

In [ ]:
# ── 10. ZIP Paketle (İndir / unilabellm'e Yükle) ─────────────────────────────
import zipfile, time

zip_path = Path('/kaggle/working') / f'auto_labeled_{int(time.time())}.zip'

print(f'📦 Paketleniyor → {zip_path.name} ...')
file_list = list(OUTPUT_DIR.rglob('*'))
files_to_zip = [f for f in file_list if f.is_file()]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for f in files_to_zip:
        zf.write(f, f.relative_to(OUTPUT_DIR))

size_mb = zip_path.stat().st_size / 1e6
print(f'✅ Hazır: {zip_path.name}  ({size_mb:.1f} MB)')
print(f'   → Kaggle Output sekmesinden indir')
print(f'   → unilabellm\'e yükleyip gözden geçir')

## Sonraki Adımlar

1. **ZIP'i indir** → Kaggle Output sekmesi
2. **unilabellm'e yükle** → New Session → dataset path olarak ZIP'i aç
3. **Gözden geçir** → Sample Viewer'da hatalı labelları düzenle (Label Editor)
4. **Export** → YOLO formatında temiz veri seti
5. **Yeniden eğit** → `unilabellm_training.ipynb` ile modeli güçlendir

---
> **İpucu:** `CONF_THRESHOLD = 0.30` ile başla. Çok gürültülüyse 0.40-0.50'ye çıkar.
> Boş görüntüleri dahil etmek "background" örnekleri olarak modeli güçlendirir — silme.